# Multi-Agent GDP Chart (Fixed)

In [1]:
import os
from typing import Annotated
import traceback

from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_experimental.utilities import PythonREPL
from langgraph_supervisor import create_supervisor
from langgraph.prebuilt import create_react_agent

load_dotenv()

True

In [2]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing. Add it to your environment or .env file.")
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY is missing. Add it to your environment or .env file.")

In [3]:
repl = PythonREPL()

@tool
def python_repl_tool(
    code: Annotated[str, "Python code to execute. For charts, save image files and print the path."],
):
    """Execute Python code in REPL and return stdout/errors."""
    try:
        result = repl.run(code)
        stdout = result if result else "(no stdout)"
        return f"Executed successfully.\nStdout:\n{stdout}"
    except Exception:
        return f"Execution failed:\n{traceback.format_exc()}"

In [4]:
search_tool = TavilySearchResults(tavily_api_key=TAVILY_API_KEY)
llm = ChatOpenAI(model="gpt-4o")

C:\Users\DELL\AppData\Local\Temp\ipykernel_18296\1890212280.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(tavily_api_key=TAVILY_API_KEY)


In [5]:
research_agent = create_react_agent(
    model=llm,
    name="research_agent",
    tools=[search_tool],
    prompt=(
        "You are a research-only agent.\n"
        "Find reliable GDP values for the UK for the requested years and return concise numeric data.\n"
        "Do not generate code or charts."
    ),
)

C:\Users\DELL\AppData\Local\Temp\ipykernel_18296\676392544.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(


In [6]:
code_agent = create_react_agent(
    model=llm,
    name="code_agent",
    tools=[python_repl_tool],
    prompt=(
        "You are a code-only agent.\n"
        "If the user asks for a chart, you MUST call python_repl_tool.\n"
        "Do not refuse and do not ask user to run code manually.\n"
        "Generate matplotlib code with Agg backend, save chart to 'uk_gdp.png', and print the saved filename.\n"
        "Use only data provided in the conversation context."
    ),
)

C:\Users\DELL\AppData\Local\Temp\ipykernel_18296\3574259523.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  code_agent = create_react_agent(


In [7]:
workflow = create_supervisor(
    [research_agent, code_agent],
    model=llm,
    prompt=(
        "You are a supervisor managing two agents.\n"
        "Use research_agent for facts/data.\n"
        "Use code_agent for any plotting/coding request.\n"
        "For chart requests, always route to code_agent after data is collected."
    ),
)

app = workflow.compile()

In [8]:
result = app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "what's the GDP of UK in past three years, draw a line chart?",
        }
    ]
})

result

Python REPL can execute arbitrary code. Use with caution.


{'messages': [HumanMessage(content="what's the GDP of UK in past three years, draw a line chart?", additional_kwargs={}, response_metadata={}, id='79a22199-3dfa-4d40-882e-08f4f8c22943'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 117, 'total_tokens': 130, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_16396ea983', 'id': 'chatcmpl-DEWQNFcSGYW0FsWhgvRh4FukbcJci', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='supervisor', id='lc_run--019ca881-c7ba-77e3-936d-ce7840043d74-0', tool_calls=[{'name': 'transfer_to_research_agent', 'args': {}, 'id': 'call_jnArDArCDLZ5WYKP7PITZ7rp', 'type': 'tool_call'}], invalid_tool_calls=

In [ ]:
print(abcdef)